In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from frag_gnn.runner import load_config, init_dataset, init_dataloader
from frag_gnn.pl_model import FragGNNPL
from frag_gnn.dataset import SpecMolFragDataset
from frag_gnn.utils.misc_utils import scatter_reduce, scatter_argmax, scatter_argtopk, EPS, booltype, scatter_masked_softmax, scatter_logsumexp, safelog
from frag_gnn.utils.script_utils import init_wandb_ckpt, run_inference, select_model_vals, sparse_kl, log_mean, log_std
from frag_gnn.loss import sparse_conditional_entropy_fn

import numpy as np
import torch as th
import pandas as pd
import argparse	
import wandb
import os
from pprint import pprint

In [3]:
baseline_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_baseline_test_False/vals.pkl"
baseline_vals = th.load(baseline_vals_fp)

In [4]:
low_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_posfn_test_False/vals.pkl"
low_vals = th.load(low_vals_fp)

In [5]:
high_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_negfn_test_False/vals.pkl"
high_vals = th.load(high_vals_fp)

In [6]:
mix_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_mix_test_False/vals.pkl"
mix_vals = th.load(mix_vals_fp)

In [7]:
def ensemble_vals(vals, nb=True):

    select_keys = [
        "pred_formula_logprobs",
        "pred_formula_batch_idxs",
        "pred_formula_formula_idxs",
    ]
    if nb:
        select_keys += [
            "pred_nb_joint_logprobs",
            "pred_nb_formula_node_logprobs",
            "pred_nb_joint_formula_idxs",
            "pred_nb_joint_batch_idxs",
            "pred_nb_joint_node_idxs"
        ]
    else:
        select_keys += [
            "pred_joint_logprobs",
            "pred_formula_node_logprobs",
            "pred_joint_formula_idxs",
            "pred_joint_batch_idxs",
            "pred_joint_node_idxs",
    ]

    select_d = select_model_vals(vals,select_keys,stack_dim=0)
    # formula stuff
    ens_formula_logprobs = log_mean(select_d["pred_formula_logprobs"],dim=0)
    ens_formula_batch_idxs = select_d["pred_formula_batch_idxs"][0]
    ens_formula_formula_idxs = select_d["pred_formula_formula_idxs"][0]
    
    ens_d = {
        "ens_formula_logprobs": ens_formula_logprobs,
        "ens_formula_batch_idxs": ens_formula_batch_idxs,
        "ens_formula_formula_idxs": ens_formula_formula_idxs,
    }
    
    if nb:
        
        # non-iso joint stuff
        ens_nb_joint_logprobs = log_mean(select_d["pred_nb_joint_logprobs"],dim=0)
        ens_nb_joint_formula_idxs = select_d["pred_nb_joint_formula_idxs"][0]
        ens_nb_joint_batch_idxs = select_d["pred_nb_joint_batch_idxs"][0]
        ens_nb_joint_node_idxs = select_d["pred_nb_joint_node_idxs"][0]
        ens_nb_formula_node_logprobs = ens_nb_joint_logprobs - ens_formula_logprobs[ens_nb_joint_formula_idxs]
        ens_nb_formula_node_logprobs = th.clamp(ens_nb_formula_node_logprobs, max=0.)
        ens_nb_formula_node_lse = scatter_logsumexp(ens_nb_formula_node_logprobs, ens_nb_joint_formula_idxs, dim_size=ens_formula_formula_idxs.shape[0])
    
        ens_d.update({
            "ens_nb_joint_logprobs": ens_nb_joint_logprobs,
            "ens_nb_joint_formula_idxs": ens_nb_joint_formula_idxs,
            "ens_nb_joint_batch_idxs": ens_nb_joint_batch_idxs,
            "ens_nb_joint_node_idxs": ens_nb_joint_node_idxs,
            "ens_nb_formula_node_logprobs": ens_nb_formula_node_logprobs,
            "ens_nb_formula_node_lse": ens_nb_formula_node_lse
        })

    else:

        # iso joint stuff
        ens_joint_logprobs = log_mean(select_d["pred_joint_logprobs"],dim=0)
        ens_joint_formula_idxs = select_d["pred_joint_formula_idxs"][0]
        ens_joint_batch_idxs = select_d["pred_joint_batch_idxs"][0]
        ens_joint_node_idxs = select_d["pred_joint_node_idxs"][0]
        ens_formula_node_logprobs = ens_joint_logprobs - ens_formula_logprobs[ens_joint_formula_idxs]
        ens_formula_node_logprobs = th.clamp(ens_formula_node_logprobs, max=0.)
        ens_formula_node_lse = scatter_logsumexp(ens_formula_node_logprobs, ens_joint_formula_idxs, dim_size=ens_formula_formula_idxs.shape[0])
    
        ens_d.update({
            "ens_joint_logprobs": ens_joint_logprobs,
            "ens_joint_formula_idxs": ens_joint_formula_idxs,
            "ens_joint_batch_idxs": ens_joint_batch_idxs,
            "ens_joint_node_idxs": ens_joint_node_idxs,
            "ens_formula_node_logprobs": ens_formula_node_logprobs,
            "ens_formula_node_lse": ens_formula_node_lse,
        })
    
    return ens_d

In [8]:
baseline_nb_ens_vals = ensemble_vals(baseline_vals, nb=True)


In [9]:
low_nb_ens_vals = ensemble_vals(low_vals, nb=True)

In [10]:
high_nb_ens_vals = ensemble_vals(high_vals, nb=True)

In [11]:
mix_nb_ens_vals = ensemble_vals(mix_vals, nb=True)

In [54]:
def get_topk_formula_node_idxs(ens_vals, k, nb=True, filter_thresh=0.01):

    ens_formula_logprobs = ens_vals["ens_formula_logprobs"]
    ens_formula_mask = ens_formula_logprobs > np.log(filter_thresh)

    if nb:
        prefix = "nb_"
    else:
        prefix = ""

    ens_counts = scatter_reduce(
        src=th.ones_like(ens_vals[f"ens_{prefix}formula_node_logprobs"], dtype=th.long),
        index=ens_vals[f"ens_{prefix}joint_formula_idxs"],
        reduce="sum",
        dim_size=th.max(ens_vals[f"ens_{prefix}joint_formula_idxs"])+1
    )

    ens_nb_formula_node_argtopk = scatter_argtopk(
        ens_vals[f"ens_{prefix}formula_node_logprobs"],
        ens_vals[f"ens_{prefix}joint_formula_idxs"],
        ens_vals[f"ens_{prefix}joint_node_idxs"],
        k,
        dim_size=ens_formula_mask.shape[0],
        return_max=False
    )

    return ens_formula_mask, ens_counts, ens_nb_formula_node_argtopk

    

In [55]:
k = 1

In [56]:
baseline_nb_mask, baseline_nb_counts, baseline_nb_argtopk = get_topk_formula_node_idxs(baseline_nb_ens_vals, k, nb=True)
print(th.unique(baseline_nb_counts[baseline_nb_mask], return_counts=True)[1])
print((baseline_nb_counts[baseline_nb_mask] < k).float().mean())
print((baseline_nb_argtopk[baseline_nb_mask] == -1).any(dim=1).float().mean())

tensor([28960,  9693,  5543,  3189,  1873,  1386,   838,   618,   473,   351,
          267,   213,   146,   136,   106,    99,    84,    71,    67,    46,
           38,    39,    35,    34,    38,    37,    18,    34,    34,    30,
           14,     8,     9,    14,     9,    16,     8,     9,     6,     6,
            6,     5,     4,     5,     8,     3,     2,     2,     1,     2,
            6,     3,     4,     4,     2,     1,     1,     1,     1,     2,
            1,     1,     1,     1,     1,     1,     1,     1])
tensor(0.)
tensor(0.)


In [57]:
low_nb_mask, low_nb_counts, low_nb_argtopk = get_topk_formula_node_idxs(low_nb_ens_vals, k, nb=True)
print(th.unique(low_nb_counts[low_nb_mask], return_counts=True)[1])
print((low_nb_counts[low_nb_mask] < k).float().mean())
print((low_nb_argtopk[low_nb_mask] == -1).any(dim=1).float().mean())

tensor([28860,  9538,  5342,  3113,  1838,  1346,   850,   635,   466,   348,
          268,   213,   142,   133,   116,    99,    84,    74,    65,    48,
           42,    40,    37,    38,    37,    34,    15,    36,    37,    29,
           11,     9,     9,    15,    11,    16,     9,     7,     5,     7,
            6,     6,     4,     5,     9,     3,     3,     1,     1,     2,
            5,     5,     1,     4,     3,     2,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     2,     1])
tensor(0.)
tensor(0.)


In [58]:
high_nb_mask, high_nb_counts, high_nb_argtopk = get_topk_formula_node_idxs(high_nb_ens_vals, k, nb=True)
print(th.unique(high_nb_counts[high_nb_mask], return_counts=True)[1])
print((high_nb_counts[high_nb_mask] < k).float().mean())
print((high_nb_argtopk[high_nb_mask] == -1).any(dim=1).float().mean())

tensor([28889,  9680,  5597,  3306,  1868,  1375,   881,   639,   487,   356,
          274,   211,   136,   127,   105,    89,    74,    70,    60,    41,
           40,    31,    30,    30,    30,    31,    17,    32,    30,    29,
           12,     7,     9,    13,     9,    15,     7,     5,     5,     6,
            6,     7,     3,     5,     9,     2,     2,     2,     2,     3,
            2,     2,     1,     2,     4,     1,     1,     1,     1,     1,
            2,     1,     1,     1,     1,     1,     1,     1,     1])
tensor(0.)
tensor(0.)


In [59]:
mix_nb_mask, mix_nb_counts, mix_nb_argtopk = get_topk_formula_node_idxs(mix_nb_ens_vals, k, nb=True)
print(th.unique(mix_nb_counts[mix_nb_mask], return_counts=True)[1])
print((mix_nb_counts[mix_nb_mask] < k).float().mean())
print((mix_nb_argtopk[mix_nb_mask] == -1).any(dim=1).float().mean())

tensor([28782,  9562,  5490,  3190,  1853,  1355,   840,   620,   484,   350,
          265,   224,   136,   125,   108,    99,    75,    74,    60,    43,
           39,    39,    34,    37,    34,    34,    16,    34,    36,    30,
           13,     9,    10,    15,    11,    15,     8,     6,     5,     7,
            6,     5,     4,     5,     7,     3,     2,     1,     1,     3,
            6,     3,     1,     4,     4,     2,     1,     1,     1,     1,
            2,     1,     1,     2,     1,     1,     1,     1])
tensor(0.)
tensor(0.)


In [60]:
def compare_all(masks, argtopks):

    num_ens = len(masks)
    compare_mat = th.zeros((num_ens,num_ens),dtype=th.float32)

    for i in range(num_ens):
        for j in range(num_ens):
            mask_i = masks[i]
            mask_j = masks[j]
            argtopk_i = argtopks[i]
            argtopk_j = argtopks[j]
            mask_ij = mask_i & mask_j
            equal_ij = th.all(argtopk_i[mask_ij] == argtopk_j[mask_ij], dim=1)
            compare_mat[i,j] = equal_ij.float().mean()

    return compare_mat

In [61]:
compare_mat = compare_all(
    [baseline_nb_mask, low_nb_mask, high_nb_mask, mix_nb_mask],
    [baseline_nb_argtopk, low_nb_argtopk, high_nb_argtopk, mix_nb_argtopk]
)
compare_mat

tensor([[1.0000, 0.9703, 0.9669, 0.9779],
        [0.9703, 1.0000, 0.9604, 0.9766],
        [0.9669, 0.9604, 1.0000, 0.9695],
        [0.9779, 0.9766, 0.9695, 1.0000]])

In [62]:
baseline_nb_c_mask = baseline_nb_mask & (baseline_nb_counts > k)
low_nb_c_mask = low_nb_mask & (low_nb_counts > k)
high_nb_c_mask = high_nb_mask & (high_nb_counts > k)
mix_nb_c_mask = mix_nb_mask & (mix_nb_counts > k)
print(baseline_nb_mask.float().mean(), baseline_nb_c_mask.float().mean())
print(low_nb_mask.float().mean(), low_nb_c_mask.float().mean())
print(high_nb_mask.float().mean(), high_nb_c_mask.float().mean())
print(mix_nb_mask.float().mean(), mix_nb_c_mask.float().mean())

tensor(0.0232) tensor(0.0109)
tensor(0.0230) tensor(0.0107)
tensor(0.0232) tensor(0.0110)
tensor(0.0230) tensor(0.0108)


In [63]:
compare_mat = compare_all(
    [baseline_nb_c_mask, low_nb_c_mask, high_nb_c_mask, mix_nb_c_mask],
    [baseline_nb_argtopk, low_nb_argtopk, high_nb_argtopk, mix_nb_argtopk]
)
compare_mat

tensor([[1.0000, 0.9359, 0.9289, 0.9526],
        [0.9359, 1.0000, 0.9143, 0.9496],
        [0.9289, 0.9143, 1.0000, 0.9344],
        [0.9526, 0.9496, 0.9344, 1.0000]])